# Aula 04 "K" Vizinhos Mais Próximos ("K"NN ou "k"-nearest neighbors)

## Exemplo de Sala

### Parte 01 - Criar um Dataset Balanceado

#### Importar Bibliotecas

In [ ]:
import pandas as pd
import numpy as np

##### Mandar o Python gerar valores aleatórios para um Dataset (uma tabela).
Essa tabela possui clientes de um supermercado agrupados em 5 perfis, (estratos ou grupos)

In [ ]:
# Configurar semente para reprodutibilidade dos resultados na aula
np.random.seed(42)

#### Gerar o a função ou método que cria a tabela de clientes

In [ ]:
def gerar_clientes_por_perfil(perfil, n_amostras):
    """
    Gera dados sintéticos baseados nas características de cada perfil de cliente.

    Atributos:
    - qtd_frescos: % de itens hortifrúti/frescos na cesta (0 a 100)
    - qtd_industrializados: % de itens ultraprocessados/congelados (0 a 100)
    - preco_medio_item: Valor médio gasto por item em R$
    - volume_total_itens: Quantidade total de itens no carrinho
    - %_itens_promocao: Porcentagem de itens comprados com desconto (0 a 100)
    """
    if perfil == "Cliente Saudável":
        frescos = np.random.normal(loc=70, scale=8, size=n_amostras)
        industrializados = np.random.normal(loc=10, scale=5, size=n_amostras)
        preco_medio = np.random.normal(loc=18, scale=3, size=n_amostras)
        volume = np.random.normal(loc=25, scale=5, size=n_amostras)
        promocao = np.random.normal(loc=15, scale=5, size=n_amostras)

    elif perfil == "Família Grande":
        frescos = np.random.normal(loc=30, scale=8, size=n_amostras)
        industrializados = np.random.normal(loc=45, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=12, scale=2, size=n_amostras)
        volume = np.random.normal(loc=85, scale=12, size=n_amostras)
        promocao = np.random.normal(loc=40, scale=10, size=n_amostras)

    elif perfil == "Casal Jovem Gourmet":
        frescos = np.random.normal(loc=45, scale=7, size=n_amostras)
        industrializados = np.random.normal(loc=15, scale=5, size=n_amostras)
        preco_medio = np.random.normal(loc=35, scale=5, size=n_amostras)
        volume = np.random.normal(loc=20, scale=4, size=n_amostras)
        promocao = np.random.normal(loc=10, scale=4, size=n_amostras)

    elif perfil == "Solteiro Prático":
        frescos = np.random.normal(loc=10, scale=4, size=n_amostras)
        industrializados = np.random.normal(loc=70, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=15, scale=3, size=n_amostras)
        volume = np.random.normal(loc=12, scale=3, size=n_amostras)
        promocao = np.random.normal(loc=20, scale=5, size=n_amostras)

    elif perfil == "Caçador de Ofertas":
        frescos = np.random.normal(loc=25, scale=6, size=n_amostras)
        industrializados = np.random.normal(loc=35, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=8, scale=2, size=n_amostras)
        volume = np.random.normal(loc=35, scale=8, size=n_amostras)
        promocao = np.random.normal(loc=80, scale=8, size=n_amostras)

    df_perfil = pd.DataFrame({
        'pct_frescos': np.clip(frescos, 0, 100),
        'pct_industrializados': np.clip(industrializados, 0, 100),
        'preco_medio_item': np.clip(preco_medio, 1, None),
        'volume_total_itens': np.clip(volume, 1, None).astype(int),
        'pct_promocao': np.clip(promocao, 0, 100),
        'perfil_cliente': perfil
    })

    return df_perfil

#### Crie uma coluna contendo os 5 perfis (estratos ou grupos)

In [ ]:
# Gerar 20 amostras para cada um dos 5 perfis (Total: 100 clientes)
perfis = ["Cliente Saudável", "Família Grande", "Casal Jovem Gourmet", "Solteiro Prático", "Caçador de Ofertas"]

#### adicione a coluna que você criou na tabela que o python havia gerado anteriormente

In [ ]:
dfs = [gerar_clientes_por_perfil(p, n_amostras=20) for p in perfis]acima

#### Coloque agora a tabela de clientes contendo a coluna com o nome dos perfis de cada cliente em formato DataFrame ( "tabela" python )

In [ ]:
# Combinar em um único DataFrame
df_clientes = pd.concat(dfs, ignore_index=True)

#### Deixando apenas as casas dos centavos (duas casas depois da virgula)

In [ ]:
# Arredondar valores numéricos para melhor apresentação
df_clientes = df_clientes.round(2)

### Exibir os 5 primeiros registros e salvar em CSV

In [ ]:
print(df_clientes.head())

#### Salvar no Disco

In [ ]:
df_clientes.to_csv("clientes_supermercado_knn.csv", index=False)

### Parte 02 -

#### Importando as Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

#### Carregar do disco Tabela criada anteriormente na Parte 01

In [ ]:
# 1. Carregar o dataset salvo anteriormente (ou definir a variável df_clientes)
df_clientes = pd.read_csv("clientes_supermercado_knn.csv")

### Separar entrada (X) e saída ou rótulo (Y)

In [ ]:
# 2. Separar Atributos (X) e Rótulo (y)
X = df_clientes.drop(columns=['perfil_cliente'])
y = df_clientes['perfil_cliente']

### Normalizar valores (colocar todos os valores nas mesmas escalas ou unidades)

In [ ]:
# 3. Normalização/Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#### Criar seu classificador no padrão algoritmo "KNN" . Salve esse modelo na variável **knn**

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

#### Treinar seu modelo: submeter todos os valores da tabela ao modelo aos 5 vizinhos mais próximos
Lembre-se que você tinha 5 estratos (5 grupos de consumidores).

In [ ]:
# 4. Treinar o KNN
knn.fit(X_scaled, y)

### Inserir um novo cliente na tabela de 100 clientes originais (com 5 grupos contendo 20 clientes enquadradaos naquele perfil)

In [ ]:
# Exemplo: %_frescos, %_industrializados, preco_medio, volume, %_promocao
novo_cliente = np.array([[12, 68, 14.50, 10, 15]])

#### Coloque este novo cliente na mesma escala de valores dos outros clientes previamente existentes na tabela.

In [ ]:
novo_cliente_scaled = scaler.transform(novo_cliente)

### Etapa de Teste

#### Faça agora o modelo existente na variável "knn" calcular a qual grupo o novo cliente mais se parece usando distância euclidiana.

In [ ]:
predicao = knn.predict(novo_cliente_scaled)


#### Escreva na tela o nome do grupo ao qual o novo cliente vai ser inserido:

In [ ]:
print(f"Perfil previsto para o novo cliente: {predicao[0]}")